# Example 19: Baryonic Tully–Fisher Relation (Four Surveys)

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

The Baryonic Tully–Fisher Relation (BTFR) is a tight empirical scaling law
relating a galaxy's total baryonic mass to its asymptotic rotation velocity:

$$M_{\\rm bar} \\propto V_{\\rm flat}^4$$

This notebook demonstrates BTFR retrieval across all four surveys in the
unified corpus: **SPARC**, **THINGS**, **LITTLE THINGS**, and **WALLABY DR2**.

**Mass estimation:**
- **SPARC:** $M_{\\rm bar}$ computed from the rotation-curve decomposition at the
  outermost measured ring: $M_{\\rm gas} = 1.33\\,V_{\\rm gas}^2 R/G$ (HI + He correction),
  $M_{\\rm star} = \\Upsilon_{\\rm disk}\\,V_{\\rm disk}^2 R/G$, $M_{\\rm bul} = V_{\\rm bul}^2 R/G$.
- **THINGS / LITTLE THINGS / WALLABY:** $V_{\\rm flat}$ from `vrot_max_kms`;
  $M_{\\rm bar}$ estimated via the McGaugh (2012) BTFR calibration
  ($\\log M_{\\rm bar} = 4\\log V_{\\rm flat} + \\log 47$) as a cross-survey consistency check.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv`
and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer.
The CSV contains every kinematic parameter published by Lelli et al. (2016)
including per-galaxy inclination, distance uncertainties, mass-to-light ratios,
and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas,
Vdisk, Vbul, errV at every radial point.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**References:** McGaugh et al. (2000), ApJ 533, L99; McGaugh (2012), AJ 143, 40;
Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, json (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json':
            'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")

In [ ]:
import json
import numpy as np

# ── Load corpus ────────────────────────────────────────────────
with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

galaxies = corpus['galaxies']
print(f"Total galaxies in corpus: {len(galaxies)}")

# ── Constants ──────────────────────────────────────────────────
# G in units: kpc (km/s)^2 Msun^-1
G_KPC = 4.302e-6

# ── SPARC: compute M_bar from rotation-curve decomposition ─────
def sparc_mbar(g):
    """
    Estimate M_bar and V_flat from the outermost rotation-curve ring.
    Returns (log10(M_bar/Msun), V_flat [km/s]) or (None, None).
    """
    m2l = g.get('m2l_disk')
    if not m2l or not g.get('data'):
        return None, None
    m2l = float(m2l)
    ring = g['data'][-1]          # outermost ring
    R    = ring.get('Rad')
    Vgas = ring.get('Vgas', 0)
    Vdisk = ring.get('Vdisk', 0)
    Vbul  = ring.get('Vbul',  0)
    Vobs  = ring.get('Vobs')
    if not R or not Vobs:
        return None, None
    R, Vgas, Vdisk, Vbul, Vobs = (
        float(R), float(Vgas), float(Vdisk), float(Vbul), float(Vobs))
    Mgas  = 1.33 * max(Vgas,  0)**2 * R / G_KPC   # HI + He
    Mstar = m2l  * Vdisk**2          * R / G_KPC   # stellar disk
    Mbul  =        Vbul**2           * R / G_KPC   # bulge
    Mbar  = Mgas + Mstar + Mbul
    if Mbar <= 0:
        return None, None
    return np.log10(Mbar), Vobs

def get_vmax(g):
    """Return V_flat from summary field or max(Vrot) over rings."""
    v = g.get('vrot_max_kms') or g.get('Vobs_max_kms')
    if v:
        try: return float(v)
        except: pass
    if g.get('data'):
        vcol = 'Vrot' if 'Vrot' in (g.get('columns') or {}) else 'Vobs'
        vals = [float(r[vcol]) for r in g['data'] if r.get(vcol)]
        if vals:
            return max(vals)
    return None

# ── Collect per-survey data points ────────────────────────────
results = {s: [] for s in ('SPARC', 'THINGS', 'LITTLE_THINGS', 'WALLABY')}

for g in galaxies:
    survey = g.get('survey')
    if survey not in results:
        continue
    if survey == 'SPARC':
        logM, V = sparc_mbar(g)
        if logM is not None and V > 0:
            results['SPARC'].append((V, logM))
    else:
        vmax = get_vmax(g)
        if not vmax or vmax <= 0:
            continue
        # McGaugh (2012) calibration: log M_bar = 4 log V + log(47)
        logM = 4.0 * np.log10(vmax) + np.log10(47.0)
        results[survey].append((vmax, logM))

for survey, pts in results.items():
    print(f"  {survey:<15}: {len(pts):>3} galaxies")

In [ ]:
import matplotlib.pyplot as plt

# ── Style ──────────────────────────────────────────────────────
COLORS  = {'SPARC':'#1f77b4', 'THINGS':'#ff7f0e',
           'LITTLE_THINGS':'#2ca02c', 'WALLABY':'#d62728'}
MARKERS = {'SPARC':'o', 'THINGS':'s', 'LITTLE_THINGS':'^', 'WALLABY':'D'}
LABELS  = {
    'SPARC':         'SPARC — $M_{\\rm bar}$ from decomposition',
    'THINGS':        'THINGS — $V^4$ proxy',
    'LITTLE_THINGS': 'LITTLE THINGS — $V^4$ proxy',
    'WALLABY':       'WALLABY DR2 — $V^4$ proxy',
}

fig, ax = plt.subplots(figsize=(8, 6))

for survey in ('SPARC', 'THINGS', 'LITTLE_THINGS', 'WALLABY'):
    pts = results[survey]
    if not pts:
        continue
    vs, ms = zip(*pts)
    label = f"{LABELS[survey]} (n={len(pts)})"
    ax.scatter(np.log10(vs), ms,
               s=25, alpha=0.7, edgecolors='none',
               color=COLORS[survey], marker=MARKERS[survey],
               label=label)

# McGaugh (2012) reference BTFR
v_ref = np.logspace(0.8, 2.7, 200)
ax.plot(np.log10(v_ref), 4*np.log10(v_ref) + np.log10(47),
        'k--', lw=1.2, alpha=0.55, label='McGaugh (2012) calibration')

ax.set_xlabel(r'$\log_{10}\,V_{\rm flat}$ (km s$^{-1}$)', fontsize=13)
ax.set_ylabel(r'$\log_{10}\,M_{\rm bar}$ ($M_\odot$)',    fontsize=13)
ax.set_title(
    'Baryonic Tully–Fisher Relation\n'
    'Unified HI Corpus v7.0 — SPARC · THINGS · LITTLE THINGS · WALLABY DR2',
    fontsize=11)
ax.legend(fontsize=9, framealpha=0.9)
ax.set_xlim(0.9, 2.8)
ax.set_ylim(6.5, 12.5)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('ex19_btfr_four_surveys.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ex19_btfr_four_surveys.png")